# rPPG · 第 0 格环境 + 数据

在 VS Code 里依次点运行这三格即可。第 0 格只在**换机器/换硬件**时会有实际动作，其余时候全是"跳过"。

In [ ]:
# ============ 第 0 格：环境（固定块，永远不要改）============
# 幂等：挂 Drive → 拉代码 → 判新 VM → 装依赖/打补丁 → 载凭据 → 备数据
import os, subprocess, sys
from pathlib import Path

repo = Path("/content/colab-workbench")
if not repo.exists():
    subprocess.run(["git", "clone", "-q",
        "https://github.com/Elisabethhui/colab-workbench.git",
        str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=False)

# 同一个内核里重跑时，必须丢掉旧的模块缓存，否则改动不生效
for _m in [m for m in sys.modules if m == "bootstrap" or m.startswith("projects")]:
    del sys.modules[_m]
sys.path.insert(0, str(repo))

from bootstrap import setup
ctx = setup("rppg")

## 第 1 格 · 检查数据到位没有

只读，不发网络请求。

In [ ]:
# 数据在 Drive：Colab_Projects/_data/kaggle/<owner>__<slug>
import glob, json
from pathlib import Path

root = Path(ctx.paths.datasets) / "kaggle"
for d in sorted(root.glob("*/")):
    vids = glob.glob(str(d / "**" / "subject*_vid.avi"), recursive=True)
    mf = d / ".manifest.json"
    info = json.loads(mf.read_text()) if mf.exists() else {}
    print(f"{d.name}")
    print(f"   视频 {len(vids)} 个 | manifest: {'有' if info else '无'} | 记录文件数 {info.get('files', '-')}")

view = Path("/content/UBFC")
print("\n训练用的软链视图:", view, "存在" if view.exists() else "不存在")
if view.exists():
    subs = [p.name for p in view.iterdir() if p.is_dir()]
    print(f"   拼接了 {len(subs)} 个 subject:", ", ".join(sorted(subs)[:5]), "..." if len(subs) > 5 else "")

## 第 2 格 · 业务格示例（产物写回 Drive，换机器还在）

In [ ]:
import time, json
from pathlib import Path

out = Path(ctx.paths.outputs) / f"smoke_{time.strftime('%Y%m%d_%H%M%S')}.json"
out.write_text(json.dumps({
    "project": ctx.project,
    "ok": True,
    "drive_data": str(ctx.paths.datasets),
    "repo": str(ctx.paths.repo),
}, ensure_ascii=False, indent=2))
print("已写入 Drive:", out)
print(out.read_text())